# 06 - Check that the HPO Database is Populated

## Learning objectives
- Connect to the `hpo` database in Neo4j from Python.
- Verify that the graph has been populated with nodes and relationships.
- Inspect label distribution, relationship types, and sample data.

In [1]:
from neo4j import GraphDatabase

In [2]:
# Connection settings.
uri = "bolt://localhost:7687"
user = "neo4j"
password = "12345678"
database = "hpo"

driver = GraphDatabase.driver(uri, auth=(user, password))
driver.verify_connectivity()
print(f"Connected to Neo4j at {uri}")

Connected to Neo4j at bolt://localhost:7687


## Step 1 - Total Node and Relationship Counts

A quick sanity check: if both counts are greater than zero the database has been populated.

In [3]:
with driver.session(database=database) as session:
    total_nodes = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    total_rels  = session.run("MATCH ()-[r]->() RETURN count(r) AS cnt").single()["cnt"]

print(f"Total nodes:         {total_nodes:,}")
print(f"Total relationships: {total_rels:,}")

if total_nodes == 0:
    print("\n⚠️  The database appears EMPTY. Run notebook 05 first to import the HPO ontology.")
else:
    print("\n✅ The database is populated.")

Total nodes:         1
Total relationships: 0

✅ The database is populated.


## Step 2 - Label Distribution

Count nodes per label to understand the composition of the graph.

In [4]:
with driver.session(database=database) as session:
    label_dist = session.run(
        "MATCH (n) "
        "UNWIND labels(n) AS lbl "
        "RETURN lbl AS label, count(*) AS count "
        "ORDER BY count DESC"
    ).data()

print(f"{'Label':<30} {'Count':>10}")
print("-" * 42)
for row in label_dist:
    print(f"{row['label']:<30} {row['count']:>10,}")

Label                               Count
------------------------------------------
_GraphConfig                            1


## Step 3 - Relationship Type Distribution

Count relationships per type to see how entities are connected.

In [5]:
with driver.session(database=database) as session:
    rel_dist = session.run(
        "MATCH ()-[r]->() "
        "RETURN type(r) AS relType, count(*) AS count "
        "ORDER BY count DESC"
    ).data()

print(f"{'Relationship Type':<40} {'Count':>10}")
print("-" * 52)
for row in rel_dist:
    print(f"{row['relType']:<40} {row['count']:>10,}")

Relationship Type                             Count
----------------------------------------------------


## Step 4 - Schema Overview (Constraints & Indexes)

Verify that the expected constraints and indexes are in place.

In [6]:
with driver.session(database=database) as session:
    print("=== Constraints ===")
    constraints = session.run("SHOW CONSTRAINTS").data()
    if not constraints:
        print("  (none)")
    for c in constraints:
        print(f"  {c['name']}  |  type: {c['type']}  |  entity: {c['entityType']}  |  properties: {c['properties']}")

    print("\n=== Indexes ===")
    indexes = session.run("SHOW INDEXES").data()
    if not indexes:
        print("  (none)")
    for idx in indexes:
        print(f"  {idx['name']}  |  type: {idx['type']}  |  entity: {idx['entityType']}  |  properties: {idx['properties']}")

=== Constraints ===
  constraint_e35c7fe  |  type: UNIQUENESS  |  entity: NODE  |  properties: ['id']
  n10s_unique_uri  |  type: UNIQUENESS  |  entity: NODE  |  properties: ['uri']

=== Indexes ===
  constraint_e35c7fe  |  type: RANGE  |  entity: NODE  |  properties: ['id']
  disease_id  |  type: RANGE  |  entity: NODE  |  properties: ['id']
  index_343aff4e  |  type: LOOKUP  |  entity: NODE  |  properties: None
  index_f7700477  |  type: LOOKUP  |  entity: RELATIONSHIP  |  properties: None
  n10s_unique_uri  |  type: RANGE  |  entity: NODE  |  properties: ['uri']
  phenotype_id  |  type: RANGE  |  entity: NODE  |  properties: ['id']


## Step 5 - Sample Nodes

Retrieve a few sample nodes to visually inspect the data.

In [7]:
with driver.session(database=database) as session:
    samples = session.run(
        "MATCH (n) "
        "RETURN labels(n) AS labels, n.uri AS uri, n.label AS label, n.id AS id "
        "LIMIT 10"
    ).data()

print(f"{'Labels':<30} {'id':<20} {'label':<50} {'uri'}")
print("-" * 130)
for s in samples:
    lbl_str = ", ".join(s["labels"]) if s["labels"] else ""
    print(f"{lbl_str:<30} {str(s['id'] or ''):<20} {str(s['label'] or ''):<50} {s['uri'] or ''}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `label` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=55, offset=54>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 54, 'line': 1, 'column': 55}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n) RETURN labels(n) AS labels, n.uri AS uri, n.label AS label, n.id AS id LIMIT 10'


Labels                         id                   label                                              uri
----------------------------------------------------------------------------------------------------------------------------------
_GraphConfig                                                                                           


## Step 6 - Sample Relationships

Retrieve a few sample relationships to inspect connectivity.

In [8]:
with driver.session(database=database) as session:
    sample_rels = session.run(
        "MATCH (a)-[r]->(b) "
        "RETURN labels(a) AS from_labels, type(r) AS rel_type, labels(b) AS to_labels, "
        "       a.label AS from_label, b.label AS to_label "
        "LIMIT 10"
    ).data()

for rel in sample_rels:
    src = rel["from_label"] or ", ".join(rel["from_labels"])
    tgt = rel["to_label"] or ", ".join(rel["to_labels"])
    print(f"  ({src}) -[:{rel['rel_type']}]-> ({tgt})")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `label` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=107, offset=106>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 106, 'line': 1, 'column': 107}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (a)-[r]->(b) RETURN labels(a) AS from_labels, type(r) AS rel_type, labels(b) AS to_labels,        a.label AS from_label, b.label AS to_label LIMIT 10'
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `label` does not exist. Verify 

## Step 7 - Enrich Nodes with `HpoPhenotype` Label and Phenotype ID

Resource nodes whose URI starts with `http://purl.obolibrary.org/obo/HP` represent HPO phenotype
concepts. We add the `HpoPhenotype` label and compute an `id` property (e.g. `HP:0000001`) from
the original URI by stripping the namespace prefix and replacing `_` with `:`.

In [9]:
enrich_query = (
    'MATCH (n:Resource) '
    'WHERE n.uri STARTS WITH "http://purl.obolibrary.org/obo/HP" '
    "SET n:HpoPhenotype, "
    "    n.id = coalesce(n.id, replace(apoc.text.replace(n.uri,'(.*)obo/',''),'_', ':')) "
    "RETURN count(n) AS enriched"
)

with driver.session(database=database) as session:
    result = session.execute_write(lambda tx: tx.run(enrich_query).single())
    enriched = result["enriched"]

print(f"Enriched {enriched:,} Resource nodes with label :HpoPhenotype and phenotype id.")

Enriched 0 Resource nodes with label :HpoPhenotype and phenotype id.


In [10]:
# Quick verification: show a few enriched HpoPhenotype nodes.
with driver.session(database=database) as session:
    samples = session.run(
        "MATCH (n:HpoPhenotype) "
        "RETURN n.id AS id, n.label AS label, n.uri AS uri "
        "LIMIT 10"
    ).data()

print(f"{'id':<15} {'label':<50} {'uri'}")
print("-" * 120)
for s in samples:
    print(f"{s['id'] or '':<15} {s['label'] or '':<50} {s['uri'] or ''}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `label` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=45, offset=44>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 44, 'line': 1, 'column': 45}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n:HpoPhenotype) RETURN n.id AS id, n.label AS label, n.uri AS uri LIMIT 10'


id              label                                              uri
------------------------------------------------------------------------------------------------------------------------


In [11]:
driver.close()
print("Neo4j driver closed.")


Neo4j driver closed.
